In [12]:
import cv2
import numpy as np
import os

# -----------------------------
# File paths
# -----------------------------

haar_file = r"C:\Users\ds512\OneDrive\Documents\open_cv\haarcascade_frontalface_default.xml"
datasets = r"C:\face_datasets"

# Face image size
(width, height) = (130, 100)

print("Recognizing Face...")
print("Dataset location:", datasets)

# -----------------------------
# Check Haar Cascade
# -----------------------------

if not os.path.exists(haar_file):
    print("ERROR: Haar Cascade file not found!")
    print(haar_file)
    raise SystemExit

face_cascade = cv2.CascadeClassifier(haar_file)

if face_cascade.empty():
    print("ERROR: Could not load Haar Cascade.")
    raise SystemExit

# -----------------------------
# Check dataset
# -----------------------------

if not os.path.exists(datasets):
    print("ERROR: Dataset folder not found!")
    print(datasets)
    raise SystemExit

# -----------------------------
# Create lists
# -----------------------------

images = []
labels = []
names = {}

person_id = 0

# -----------------------------
# Read dataset
# -----------------------------

for person_name in os.listdir(datasets):

    person_path = os.path.join(datasets, person_name)

    # Skip files
    if not os.path.isdir(person_path):
        continue

    names[person_id] = person_name

    print(f"\nPerson ID: {person_id}")
    print(f"Name: {person_name}")

    for filename in os.listdir(person_path):

        image_path = os.path.join(person_path, filename)

        # Read image
        image = cv2.imread(
            image_path,
            cv2.IMREAD_GRAYSCALE
        )

        # Skip invalid images
        if image is None:
            print("Could not read:", image_path)
            continue

        # Resize image
        image = cv2.resize(
            image,
            (width, height)
        )

        images.append(image)
        labels.append(person_id)

        print("Loaded:", filename)

    person_id += 1

# -----------------------------
# Check dataset
# -----------------------------

if len(images) == 0:
    print("\nERROR: No images found!")
    raise SystemExit

print("\nTotal images:", len(images))
print("Names:", names)

# -----------------------------
# Convert to NumPy arrays
# -----------------------------

images = np.array(images)
labels = np.array(labels)

# -----------------------------
# Create LBPH recognizer
# -----------------------------

if not hasattr(cv2, "face"):
    print("\nERROR: cv2.face is not available.")
    print("Install opencv-contrib-python:")
    print("pip install opencv-contrib-python")
    raise SystemExit

model = cv2.face.LBPHFaceRecognizer_create()

# -----------------------------
# Train model
# -----------------------------

print("\nTraining model...")

model.train(images, labels)

print("Training completed successfully!")

Recognizing Face...
Dataset location: C:\face_datasets

Person ID: 0
Name: ashutosh
Loaded: 1.png
Loaded: 10.png
Loaded: 11.png
Loaded: 12.png
Loaded: 13.png
Loaded: 14.png
Loaded: 15.png
Loaded: 16.png
Loaded: 17.png
Loaded: 18.png
Loaded: 19.png
Loaded: 2.png
Loaded: 20.png
Loaded: 21.png
Loaded: 22.png
Loaded: 23.png
Loaded: 24.png
Loaded: 25.png
Loaded: 26.png
Loaded: 27.png
Loaded: 28.png
Loaded: 29.png
Loaded: 3.png
Loaded: 30.png
Loaded: 4.png
Loaded: 5.png
Loaded: 6.png
Loaded: 7.png
Loaded: 8.png
Loaded: 9.png

Person ID: 1
Name: dhruv
Loaded: 1.png
Loaded: 10.png
Loaded: 11.png
Loaded: 12.png
Loaded: 13.png
Loaded: 14.png
Loaded: 15.png
Loaded: 16.png
Loaded: 17.png
Loaded: 18.png
Loaded: 19.png
Loaded: 2.png
Loaded: 20.png
Loaded: 21.png
Loaded: 22.png
Loaded: 23.png
Loaded: 24.png
Loaded: 25.png
Loaded: 26.png
Loaded: 27.png
Loaded: 28.png
Loaded: 29.png
Loaded: 3.png
Loaded: 30.png
Loaded: 4.png
Loaded: 5.png
Loaded: 6.png
Loaded: 7.png
Loaded: 8.png
Loaded: 9.png

Person 

In [13]:
# -----------------------------
# Open webcam
# -----------------------------

webcam = cv2.VideoCapture(0)

if not webcam.isOpened():
    print("ERROR: Could not open webcam.")
    raise SystemExit

print("Webcam started.")
print("Look at the camera.")
print("Press ESC to stop.")

while True:

    # Read frame
    ret, frame = webcam.read()

    if not ret:
        print("ERROR: Could not read webcam.")
        break

    # Convert to grayscale
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(50, 50)
    )

    # -----------------------------
    # Process faces
    # -----------------------------

    for (x, y, w, h) in faces:

        # Crop face
        face = gray[y:y+h, x:x+w]

        # Resize
        face_resize = cv2.resize(
            face,
            (width, height)
        )

        # Predict
        prediction = model.predict(face_resize)

        predicted_id = prediction[0]
        confidence = prediction[1]

        # -----------------------------
        # Recognition
        # -----------------------------

        if confidence < 80:

            name = names[predicted_id]

            text = f"{name} - {confidence:.0f}"

            color = (0, 255, 0)

        else:

            text = "Not Recognized"

            color = (0, 0, 255)

        # Draw rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            color,
            2
        )

        # Display name
        cv2.putText(
            frame,
            text,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    # -----------------------------
    # Display camera
    # -----------------------------

    cv2.imshow(
        "Face Recognition - Press ESC",
        frame
    )

    # Read keyboard
    key = cv2.waitKey(10) & 0xFF

    # ESC = 27
    if key == 27:
        break

# -----------------------------
# Close camera
# -----------------------------

webcam.release()
cv2.destroyAllWindows()

print("Face recognition stopped.")

Webcam started.
Look at the camera.
Press ESC to stop.
Face recognition stopped.
